In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageEnhance, ImageFilter
from scipy.ndimage import binary_dilation
import random
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import tqdm

class RescueNetDataset(Dataset):
    """
    RescueNet Semantic Segmentation Dataset

    For each sample returns a dict with 3 tensors:
      image          → [3,  512, 512]  float32  (0-1 range, no ImageNet norm — training from scratch)
      classification → [512, 512]      int64    (class index 0-11)
      boundary       → [1,  128, 128]  float32  (0=interior 1=boundary)
    """

    CLASS_NAMES = {
        0: 'Background',
        1: 'Water',
        2: 'Building_No_Damage',
        3: 'Building_Minor_Damage',
        4: 'Building_Major_Damage',
        5: 'Building_Total_Destruction',
        6: 'Vehicle',
        7: 'Road-Clear',
        8: 'Road-Blocked',
        9: 'Tree',
        10: 'Pool'
    }

    NUM_CLASSES = 11

    def __init__(
        self,
        root,
        split           = 'train',
        image_size      = (512, 512),
        augment         = True,
        dilation_radius = 2,
    ):
        self.root            = root
        self.split           = split
        self.image_size      = image_size
        self.augment         = augment and (split == 'train')
        self.dilation_radius = dilation_radius

        self.img_dir  = os.path.join(root, split, f'{split}-org-img')
        self.mask_dir = os.path.join(root, split, f'{split}-label-img')

        self.img_files  = sorted(os.listdir(self.img_dir))
        self.mask_files = sorted(os.listdir(self.mask_dir))

        assert len(self.img_files) == len(self.mask_files), \
            f"Mismatch: {len(self.img_files)} images but {len(self.mask_files)} masks"

        print(f"[RescueNetDataset] split={split:5s} | "
              f"samples={len(self.img_files)} | "
              f"augment={self.augment}")

    def __len__(self):
        return len(self.img_files)

    def _load(self, idx):
        img_path  = os.path.join(self.img_dir,  self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert('RGB')

        mask = Image.open(mask_path)
        if mask.mode != 'P':
            mask = mask.convert('L')

        image = image.resize(self.image_size, Image.LANCZOS)
        mask  = mask.resize(self.image_size,  Image.NEAREST)

        return image, mask

    def _random_crop(self, image, mask, crop_scale=(0.5, 1.0)):
        w, h    = image.size
        scale   = random.uniform(*crop_scale)
        crop_w  = int(w * scale)
        crop_h  = int(h * scale)
        x       = random.randint(0, w - crop_w)
        y       = random.randint(0, h - crop_h)

        image = image.crop((x, y, x + crop_w, y + crop_h))
        mask  = mask.crop((x, y, x + crop_w, y + crop_h))

        image = image.resize((w, h), Image.LANCZOS)
        mask  = mask.resize((w, h), Image.NEAREST)

        return image, mask

    def _augment(self, image, mask):
        image, mask = self._random_crop(image, mask)

        # --- Existing flips + rotations (keep as-is) ---
        if random.random() > 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask  = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() > 0.5:
            image = image.transpose(Image.FLIP_TOP_BOTTOM)
            mask  = mask.transpose(Image.FLIP_TOP_BOTTOM)
        k = random.randint(0, 3)
        if k > 0:
            image = image.transpose({1:Image.ROTATE_90,2:Image.ROTATE_180,3:Image.ROTATE_270}[k])
            mask  = mask.transpose({1:Image.ROTATE_90,2:Image.ROTATE_180,3:Image.ROTATE_270}[k])

        # --- NEW: Gaussian blur (simulates sensor/altitude blur) ---
        if random.random() > 0.4:
            radius = random.uniform(0.5, 1.5)
            image = image.filter(ImageFilter.GaussianBlur(radius=radius))

        # --- NEW: Grayscale (forces texture over colour reliance) ---
        if random.random() > 0.85:
            image = ImageEnhance.Color(image).enhance(0.0)

        # --- NEW: Sharpen (counters blur, helps edges) ---
        if random.random() > 0.5:
            image = ImageEnhance.Sharpness(image).enhance(random.uniform(0.5, 2.0))

        # --- Existing colour jitter (keep) ---
        if random.random() > 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.7, 1.3))
        if random.random() > 0.5:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.7, 1.3))
        if random.random() > 0.5:
            image = ImageEnhance.Color(image).enhance(random.uniform(0.7, 1.3))

        return image, mask

    def _image_to_tensor(self, image):
        img_np     = np.array(image)
        img_tensor = torch.from_numpy(img_np)
        img_tensor = img_tensor.permute(2, 0, 1)
        img_tensor = img_tensor.float() / 255.0

        # --- NORMALIZATION (add this) ---
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

        img_tensor = (img_tensor - mean) / std
        return img_tensor                        # [3, H, W] float32

    def _mask_to_classification(self, mask):
        mask_np = np.array(mask)
        return torch.from_numpy(mask_np).long()  # [H, W] int64

    def _classification_to_boundary(self, classification_tensor):
        mask_small = classification_tensor.unsqueeze(0).unsqueeze(0).float()
        mask_small = F.interpolate(mask_small, scale_factor=0.25, mode='nearest')
        mask_np    = mask_small.squeeze().numpy().astype(np.int32)

        diff_x   = np.pad(np.abs(np.diff(mask_np, axis=1)), ((0,0),(0,1)), mode='constant')
        diff_y   = np.pad(np.abs(np.diff(mask_np, axis=0)), ((0,1),(0,0)), mode='constant')
        boundary = ((diff_x > 0) | (diff_y > 0)).astype(np.float32)

        r = self.dilation_radius // 4
        if r > 0:
            structure = np.ones((r * 2 + 1, r * 2 + 1), dtype=bool)
            boundary  = binary_dilation(boundary, structure=structure).astype(np.float32)

        return torch.from_numpy(boundary).unsqueeze(0)  # [1, 128, 128] float32

    def __getitem__(self, idx):
        # step 1+2 : load and resize
        image, mask = self._load(idx)

        # step 3 : augment (train only)
        if self.augment:
            image, mask = self._augment(image, mask)

        # step 4 : image tensor
        image_tensor = self._image_to_tensor(image)

        # step 5 : classification tensor
        classification_tensor = self._mask_to_classification(mask)

        # step 6 : boundary from augmented mask — always consistent
        # boundary_tensor = self._classification_to_boundary(classification_tensor)

        return {
            'image'         : image_tensor,           # [3,  512, 512] float32
            'classification': classification_tensor,  # [512, 512]     int64
            # 'boundary'      : boundary_tensor,        # [1,  128, 128] float32
            'filename'      : self.img_files[idx],
        }

def compute_sample_weights_fast(dataset, device):
    weights = []

    class_458 = torch.tensor([4, 5, 8], device=device)

    for mask_file in tqdm(dataset.mask_files, desc="Computing sample weights"):
        mask_path = os.path.join(dataset.mask_dir, mask_file)

        mask = Image.open(mask_path)
        mask = mask.resize(dataset.image_size, Image.NEAREST)
        mask = torch.from_numpy(np.array(mask)).to(device)

        contains_458 = torch.isin(mask, class_458).any()  # 4,5,8
        contains_9   = (mask == 9).any()                  # Tree
        contains_6   = (mask == 6).any()                  # Vehicle ✅
        contains_10  = (mask == 10).any()                 # Pool

        weight = 1.0

        if contains_458: weight += 4.0   # highest
        if contains_9:   weight += 2.5   # trees
        if contains_6:   weight += 2.0   # vehicles
        if contains_10:  weight += 1.5   # pool

        weights.append(weight)

    return weights

In [3]:
import torch

class IoU:
    def __init__(self, num_classes, ignore_index=None, device="cpu"):
        self.num_classes = num_classes
        self.device = device
        self.conf_matrix = torch.zeros((num_classes, num_classes),
                                       dtype=torch.int64,
                                       device=device)

        if ignore_index is None:
            self.ignore_index = None
        elif isinstance(ignore_index, int):
            self.ignore_index = {ignore_index}
        else:
            self.ignore_index = set(ignore_index)

    def reset(self):
        self.conf_matrix.zero_()

    def _fast_hist(self, pred, target):
        pred = pred.long()
        target = target.long()

        mask = (target >= 0) & (target < self.num_classes)

        if self.ignore_index is not None:
            for idx in self.ignore_index:
                mask = mask & (target != idx)
        pred = pred[mask]
        target = target[mask]

        hist = torch.bincount(
            self.num_classes * target + pred,
            minlength=self.num_classes * self.num_classes
        ).reshape(self.num_classes, self.num_classes)

        return hist.to(self.device)

    def add(self, predicted, target):
        if predicted.dim() == 4:
            predicted = torch.argmax(predicted, dim=1)
        if target.dim() == 4:
            target = torch.argmax(target, dim=1)

        predicted = predicted.view(-1).to(self.device)
        target = target.view(-1).to(self.device)

        self.conf_matrix += self._fast_hist(predicted, target)

    def value(self):
        conf = self.conf_matrix.float().clone()

        if self.ignore_index is not None:
            for idx in self.ignore_index:
                conf[idx, :] = 0
                conf[:, idx] = 0

        tp = torch.diag(conf)
        fp = conf.sum(dim=0) - tp
        fn = conf.sum(dim=1) - tp

        iou = tp / (tp + fp + fn + 1e-10)
        miou = torch.nanmean(iou)

        return iou, miou

def compute_iou(model, loader, num_classes, device="cuda", ignore_index=None):
    model.eval()

    iou_metric = IoU(num_classes=num_classes,
                     ignore_index=ignore_index,
                     device=device)

    iou_metric.reset()

    with torch.no_grad():
        for item in tqdm(loader, desc="Computing IoU"):
            img  = item["image"].to(device)
            mask = item["classification"].to(device)

            pred = model(img)   # (N,C,H,W)

            iou_metric.add(pred, mask)

    del img, mask, pred
    return iou_metric.value()

In [5]:
%cd segmenter

c:\Users\Jaitej\Downloads\dlcv\newFiles\segmenter


In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch.utils.data import WeightedRandomSampler
from segm.model.factory import create_segmenter

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- CONFIG ----------------
version = "1"

checkpoint_path = f"segmeter_s{version}.pth"
checkpoint  = checkpoint_path if os.path.exists(checkpoint_path) else None

num_classes = 11
batch_size  = 32
workers     = 6
epochs      = 500
lr          = 3e-4

root = "../rescuenet"
# image_size = (256, 256)
image_size = (384, 384)

# ---------------- DATA ----------------
train_dataset = RescueNetDataset(root=root, split='train', image_size=image_size, augment=True)
val_dataset   = RescueNetDataset(root=root, split='val',   image_size=image_size, augment=False)


# training_weights = compute_sample_weights_fast(train_dataset, "cuda")
if f"training_weights{image_size[0]}.npy" in os.listdir("../"):
    training_weights = np.load(f"../training_weights{image_size[0]}.npy")
    print("Loaded Training weights")
else:
    training_weights = compute_sample_weights_fast(train_dataset, "cuda")
    np.save(f"../training_weights{image_size[0]}.npy", training_weights)

sampler = WeightedRandomSampler(
    weights=training_weights,
    num_samples=len(training_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          sampler=sampler,
                        #   shuffle=True,
                        #   num_workers=workers, pin_memory=True, persistent_workers=True, prefetch_factor=3
                        )

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        # num_workers=workers, pin_memory=True, persistent_workers=True
                        )

# ---------------- COLORS ----------------
CLASS_COLORS = np.array([
    [0,0,0],[61,230,250],[180,120,120],[235,255,7],[255,184,6],
    [255,0,0],[255,0,245],[140,140,140],[160,150,20],[4,250,7],[255,235,0]
], dtype=np.uint8)

def mask_to_color(mask):
    return CLASS_COLORS[mask]

# ---------------- LOSS ----------------
# weights = torch.tensor([
#     0.0972,0.4650,0.9944,0.9752,1.2114,
#     1.2534,1.8995,0.5233,1.2445,0.2080,2.1282
# ], dtype=torch.float32).to(device)

weights = torch.tensor([
    0.1, 0.5,
    1.0,
    1.2,
    1.5,
    1.8,
    1.5,
    0.5,
    2.2,   # reduce from 3.0
    0.3,
    1.8    # reduce from 2.5
]).to(device)


seg_loss = nn.CrossEntropyLoss(
    weight=weights,
)



def validate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for item in tqdm(loader, leave=False, desc="Validating: "):
            img  = item["image"].to(device)
            mask = item["classification"].to(device)

            if mask.ndim == 4:
                mask = mask.squeeze(1)

            outputs = model(img)

            seg_l = seg_loss(outputs, mask)

            total_loss += seg_l.item()
        del img, mask, outputs
    return total_loss / len(loader)

def save_val_segmentation(model, epoch, val_loader, device, save_dir="outputs", num_samples=3):
    import os
    os.makedirs(save_dir, exist_ok=True)

    model.eval()
    images = []

    with torch.no_grad():
        for item in val_loader:
            img  = item["image"].to(device)
            mask = item["classification"].to(device)

            outputs = model(img)

            # handle tuple outputs
            if isinstance(outputs, (list, tuple)):
                pred_mask = outputs[0]
            else:
                pred_mask = outputs

            pred_mask = torch.argmax(pred_mask, dim=1)

            for b in range(img.shape[0]):
                # denormalize
                mean = torch.tensor([0.485, 0.456, 0.406], device=img.device).view(3,1,1)
                std  = torch.tensor([0.229, 0.224, 0.225], device=img.device).view(3,1,1)

                img_denorm = img[b] * std + mean
                image_np = img_denorm.permute(1, 2, 0).cpu().numpy()
                image_np = np.clip(image_np, 0, 1)
                image_vis = (image_np * 255).astype(np.uint8)

                gt_mask_raw = mask[b].cpu().numpy()
                pr_mask_raw = pred_mask[b].cpu().numpy()

                gt_mask = mask_to_color(gt_mask_raw)
                pr_mask = mask_to_color(pr_mask_raw)

                # overlays
                gt_overlay = (0.6 * image_vis + 0.4 * gt_mask).astype(np.uint8)
                pr_overlay = (0.6 * image_vis + 0.4 * pr_mask).astype(np.uint8)

                images.append((image_vis, gt_overlay, pr_overlay))

            if len(images) >= num_samples:
                break

    # Plot
    fig, axs = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    titles = ["Image", "GT Mask", "Pred Mask"]

    for i in range(num_samples):
        for j in range(3):
            axs[i, j].imshow(images[i][j])
            axs[i, j].set_title(titles[j])
            axs[i, j].axis('off')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"seg_samples_epoch_{epoch}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
# ---------------- TRAIN ----------------
def train():

    model_cfg = {
        "backbone": "vit_small",
        "normalization": "vit",

        "image_size": (384, 384),
        "patch_size": 16,

        "d_model": 384,
        "n_heads": 6,
        "n_layers": 12,

        "distilled": False,

        "decoder": {
            "name": "mask_transformer",
            "n_layers": 2,
            "drop_path_rate": 0.1,
            "dropout": 0.0,
        },

        "n_cls": 11,
    }

    model = create_segmenter(model_cfg).to(device)
    total_params = sum(p.numel() for p in model.parameters())

    # trainable params
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("Total:", total_params)
    print("Trainable:", trainable_params)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )


    scaler = torch.cuda.amp.GradScaler()
    # best_miou = 0.0
    best_val = float('inf')
    start_epoch = 0

    if checkpoint is not None:
        ckpt = torch.load(checkpoint, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optim_state'])
        scheduler.load_state_dict(ckpt['sched_state'])   # was commented out before
        best_val    = ckpt.get('best_val', float("inf"))
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {start_epoch}, best val loss: {best_val:.4f}")


    print(f"Training on {device} | epochs {start_epoch}→{epochs} | classes {num_classes} | Loaded {checkpoint}")

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch_idx, item in enumerate(tqdm(train_loader, leave=False, desc="Training")):
            img  = item["image"].to(device, memory_format=torch.channels_last)
            mask = item["classification"].to(device)

            if mask.ndim==4: mask=mask.squeeze(1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast():
                # print(img.shape)
                outputs = model(img)

                loss_main = seg_loss(outputs,mask)

                loss = loss_main

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss+=loss.item()

        with torch.no_grad():
            val_loss = validate(model, val_loader)
            # _, miou = compute_iou(
            #     model,
            #     val_loader,
            #     num_classes=num_classes,
            #     device=device,
            #     ignore_index=0
            # )
            scheduler.step()

            del img, mask, outputs
            torch.cuda.empty_cache()

        current_lr = optimizer.param_groups[0]['lr']


        txt = f"Epoch {epoch} | Train {total_loss/len(train_loader):.4f} | Val {val_loss:.4f} | LR {current_lr:.7f}"# | mIoU {miou:.4f}"
        # txt = f"Epoch {epoch} | Train {total_loss/len(train_loader):.4f} | Val {val_loss:.4f} | LR {current_lr:.7f}"

        print(txt)

        log_file = f"training_segmeter_v{version}.txt"
        with open(log_file, "a") as f:
            f.write(txt + "\n")

        if epoch % 5 == 0:
            save_val_segmentation(model, epoch, val_loader, device, save_dir="outputs", num_samples=3)

        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'sched_state': scheduler.state_dict(),
                'best_val':    best_val,
            }, checkpoint_path)
            print(f"-> Saved best model (VAL: {best_val:.4f})")

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

if __name__ == "__main__":
    # train()
    print("I have commented out Training. It will hang, if you are are trying to train it on CPU. If CUDA/GPU is being used then, please uncomment, and can check the progress. Thank you")
    pass

[RescueNetDataset] split=train | samples=3595 | augment=True
[RescueNetDataset] split=val   | samples=449 | augment=False
Loaded Training weights
I have commented out Training. It will hang, if you are are trying to train it on CPU. If CUDA/GPU is being used then, please uncomment, and can check the progress. Thank you


In [7]:
model_cfg = {
        "backbone": "vit_small",
        "normalization": "vit",

        "image_size": (384, 384),
        "patch_size": 16,

        "d_model": 384,
        "n_heads": 6,
        "n_layers": 12,

        "distilled": False,

        "decoder": {
            "name": "mask_transformer",
            "n_layers": 2,
            "drop_path_rate": 0.1,
            "dropout": 0.0,
        },

        "n_cls": 11,
    }


model = create_segmenter(model_cfg).to(device)

ignore_index=0
checkpoint = "segmeter_s1.pth"

print(checkpoint)
ckpt = torch.load(checkpoint, map_location=device)
model.load_state_dict(ckpt['model_state'])
print("Weights Loaded")
test_dataset = RescueNetDataset(root=root, split='test',   image_size=image_size, augment=False)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    # pin_memory=True,
    # persistent_workers=True,
)

iou_per_class, miou = compute_iou(
    model,
    test_loader,
    num_classes=11,
    device=device,
    ignore_index=ignore_index
)

CLASS_NAMES = [
    'Background',
    'Water',
    'Building_No_Damage',
    'Building_Minor_Damage',
    'Building_Major_Damage',
    'Building_Total_Destruction',
    'Vehicle',
    'Road-Clear',
    'Road-Blocked',
    'Tree',
    'Pool'
]

print(f"\n{'Class':<30} {'IoU':>8} {'IoU %':>8}")
print("-" * 50)

count=0
s = 0
for i, name in enumerate(CLASS_NAMES):
    if i == ignore_index:
        continue
    val = iou_per_class[i].item()
    count += 1
    s += val
    print(f"{name:<30} {val:>8.4f} {val*100:>7.2f}%")

print("-" * 50)
miou = s/count
print(f"{'Mean IoU':<30} {miou:>8.4f} {miou*100:>7.2f}%")

No pretrained weights exist for this model. Using random initialization.


segmeter_s1.pth
Weights Loaded
[RescueNetDataset] split=test  | samples=450 | augment=False


Computing IoU: 100%|██████████| 225/225 [06:10<00:00,  1.65s/it]


Class                               IoU    IoU %
--------------------------------------------------
Water                            0.8599   85.99%
Building_No_Damage               0.6247   62.47%
Building_Minor_Damage            0.5156   51.56%
Building_Major_Damage            0.4954   49.54%
Building_Total_Destruction       0.7321   73.21%
Vehicle                          0.3760   37.60%
Road-Clear                       0.7970   79.70%
Road-Blocked                     0.4948   49.48%
Tree                             0.9447   94.47%
Pool                             0.5440   54.40%
--------------------------------------------------
Mean IoU                         0.6384   63.84%
